## LWIR Model Test on RGB Images

The objective of this script is to test a YOLO model trained on LWIR images on new sets of RGB test images

First, let's import the necessary libraries.

In [2]:
!pip install -U ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.8 MB/s eta 0:00:00


In [3]:
import os
import yaml
import pandas as pd
import xml.etree.ElementTree as ET
from google.colab import drive
from ultralytics import YOLO
from pathlib import Path

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


### 1. Testing Preparation

We first mount our drive to point to the folder where the testing images are located.

In [4]:
drive.mount('/content/drive')

Mounted at /content/drive


Next, we load the trained model

<div class="alert alert-block alert-info">
    
<b>Note:</b> YOLOv8 automatically saves the model on training. The saved model can be found in this path where the training script is located. *runs/detect/train/exp*/weights/*

The model is automatically named as *"best.pt"*

</div>

In [6]:
model_path = os.path.join("/content/drive/MyDrive/Drone_images/LWIR_Training/runs/detect/train/weights", "best.pt")
lwir_model = YOLO(model_path)

Next we define the path of the test images including the labels (Pascal VOC .xml labels and .txt class files)

In [7]:
test_images = "/content/drive/MyDrive/Drone_images/RGB_Training/images/test"   # folder with test images
test_labels = "/content/drive/MyDrive/Drone_images/RGB_Training/labels/test"   # folder with YOLO .txt labels
xml_labels  = "/content/drive/MyDrive/Drone_images/RGB_Training/labels/test"   # folder with XML files

### 2. Testing the LWIR Images

We make predictions using the trained model

In [8]:
pred_results = lwir_model.predict(source = test_images, imgsz = 640, save = True)


image 1/518 /content/drive/MyDrive/Drone_images/RGB_Training/images/test/jan_afternoon_0_12.jpg: 512x640 2 at_plastics, 95.4ms
image 2/518 /content/drive/MyDrive/Drone_images/RGB_Training/images/test/jan_afternoon_0_3.jpg: 512x640 1 ap_metal, 1 ap_plastic, 3 at_plastics, 6.0ms
image 3/518 /content/drive/MyDrive/Drone_images/RGB_Training/images/test/jan_afternoon_0_52.jpg: 512x640 11 at_plastics, 6.1ms
image 4/518 /content/drive/MyDrive/Drone_images/RGB_Training/images/test/jan_afternoon_0_55.jpg: 512x640 1 ap_metal, 2 at_plastics, 6.1ms
image 5/518 /content/drive/MyDrive/Drone_images/RGB_Training/images/test/jan_afternoon_10_45.jpg: 512x640 7 at_plastics, 6.2ms
image 6/518 /content/drive/MyDrive/Drone_images/RGB_Training/images/test/jan_afternoon_10_53.jpg: 512x640 1 at_plastic, 12.1ms
image 7/518 /content/drive/MyDrive/Drone_images/RGB_Training/images/test/jan_afternoon_20_18.jpg: 512x640 1 ap_metal, 2 ap_plastics, 3 at_plastics, 7.5ms
image 8/518 /content/drive/MyDrive/Drone_images/

### 3. Model Evaluation

First, we create aand save structured configuration YAML file that will help us in evaluating the model.

In [9]:
class_names = ['ap_metal', 'ap_plastic', 'at_metal', 'at_plastic']

In [10]:
data = {
    "path":"/content/drive/MyDrive/Drone_images/RGB_Training",
    "train": os.path.join("/content/drive/MyDrive/Drone_images/RGB_Training/images/train"),
    "val": os.path.join("/content/drive/MyDrive/Drone_images/RGB_Training/images/val"),
    "test": os.path.join("/content/drive/MyDrive/Drone_images/RGB_Training/images/test"),
    "names": class_names,
}

yaml_path = os.path.join("/content/drive/MyDrive/Drone_images/RGB_Testing", "eval.yaml")
with open(yaml_path, "w") as f:

    yaml.dump(data, f, default_flow_style = False)

We finally evaluate the performance of our model by printing and saving the precision, recall, 50% and 90% mean Average Precision (mAP) scores.

In [11]:
results = lwir_model.val(data = yaml_path, split = "test",
    imgsz = 640, batch = 16, save_json = True, plots = True)

Ultralytics 8.3.202 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 76.6±17.0 MB/s, size: 170.8 KB)
val: Scanning /content/drive/MyDrive/Drone_images/RGB_Training/labels/test.cache... 517 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 518/518 671.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 33/33 2.7it/s 12.2s
                   all        518       4937      0.806      0.691      0.741      0.448
              ap_metal        331        529      0.651      0.548      0.593      0.267
            ap_plastic        328        744      0.732      0.464      0.542      0.267
              at_metal        379        555      0.898        0.8      0.863      0.579
            at_plastic        505       3109      0.943      0.952      0.966       0.68
Speed: 0.5ms preprocess, 4.1ms inference, 0.0ms loss, 1.8ms postprocess per image
Saving /content/

We save the model evaluation results for future use.

In [12]:
!cp -r /content/runs/detect/val /content/drive/MyDrive/Drone_images/RGB_Testing/lwir_model_predictions_on_rgb/

And then extract the individual metrics before converting to dataframe and eventually saving the results as a csv file.

In [13]:
precision = results.box.p
recall = results.box.r
ap50 = results.box.ap50
ap5095 = results.box.ap
class_names = results.names

In [16]:
df = pd.DataFrame({
    "class": [class_names[i] for i in range(len(precision))],
    "precision": precision,
    "recall": recall,
    "map50": ap50,
    "map50_95": ap5095,
    "model": ["LWIR on RGB Images"] * len(precision)
    })

RESULTS = "/content/drive/MyDrive/Drone_images/RGB_Testing"
output_file = os.path.join(RESULTS, "lwir_model_on_rgb_images_metrics.csv")
df.to_csv(output_file, index=False)